# Advanced Deep Learning - Exercise 4 - Multidimensional Deep Learning

In this exercise, we will look at PointNet for shape classification. Specifically, we will implement augmentations for point cloud processing and parts of PointNet.

All parts you have to implement are marked with TODO.

### Note

To save GPU compute (limited on Colab), **do the implementation first on the CPU** and then switch to the GPU for training. If you run out of GPU compute, use the CPU.

To change from CPU to GPU (or back), select `Runtime -> Change runtime type -> Hardware accelerator`.



## Basic Setup

First off, let's set up a framework.
We begin by installing the dependencies.

In [ ]:
%pip install wandb
%pip install lightning
%pip install torchmetrics
%pip install plotly
%pip install scikit-learn

To connect with wandb for logging, comment out the following line.
Don't forget to comment it out afterwards as you don't need to run this on repeated executions.

In [ ]:
# !timeout 5m wandb init

Now it's time for our basic imports. 

We will use the PyTorch Lightning as framework to simplify our code.
Generally, this is fairly similar to plain PyTorch, meaning we still define the models and functions with basic PyTorch code, but now we have some wrappers that considerably simplify the rest.
In particular, Lightning already implements the whole training and evaluation loops for us, so we don't have to do this manually.
We will see below how this works exactly.

In [ ]:
import math
import random
from pathlib import Path
from typing import Any, Dict

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import plotly
import plotly.graph_objs as go
import torch
import torchmetrics
from lightning.pytorch.loggers import WandbLogger
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

import wandb

# Enable support for tensor cores, if they are available.
torch.set_float32_matmul_precision('medium')

Next, we define a helper function to build a `Trainer` object.
This is essentially what implements the training and validation loops in PyTorch lightning and allows us to control on which device and for how long to train.
It also allows us to configure experiment logging.
Here, we set up logging via WandB.

You can adapt the parameters for `L.Trainer()` below to train for more epochs, or to train on the CPU instead of the GPU.

You can adapt the parameters for `WandbLogger()` to configure logging for your WandB project.

In [ ]:
batch_size = 32


def build_trainer(
    log: bool = True,
    max_epochs: int = 5,  # This determines how long we train.
    accelerator: str = "gpu",  # This determines what we use for training (e.g. you could set it to "cpu" for testing)
    trainer_kwargs={},  # Any additional arguments you want to pass to L.Trainer()
    wandb_kwargs={},  # TODO: Set any other wandb.init() parameters that you want here.
):
    """
    Build the PyTorch Lightning Trainer.

    :param log: Enable experiment logging.
    """
    if log:
        logger = WandbLogger(
            project="adl26_exercise_04",
            log_model="all",
            save_dir="logs",
            **wandb_kwargs,
        )
    else:
        logger = False

    kwargs = {}
    if accelerator == "gpu":
        kwargs["devices"] = 1

    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator=accelerator,
        logger=logger,
        **kwargs,
        **trainer_kwargs,
    )

    return trainer

## Data

### Preparation

We will use the dataset provided by 3D ShapeNets for this exercise. Download the [dataset](http://3dvision.princeton.edu/projects/2014/3DShapeNets/) directly to the Google Colab Runtime und unzip it. It comprises polygon meshes for 10 object categories, 3,991 models for training and 908 for testing.

In [ ]:
!wget http://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet10.zip
!unzip -q ModelNet10.zip

In [ ]:
dataset_root = Path("./ModelNet10")

This dataset consists of **.off** files that contain polygon meshes represented by *vertices* and *triangular faces*. Vertices are the 3D corner points where the edges of a polygon mesh intersect, defining its shape. Faces are the individual polygons formed by connecting these vertices, creating the surfaces of the mesh.

We will need a function to read this type of files:Next, we will build a class map, mapping between class names and 

In [ ]:
def read_off(file):
    if 'OFF' != file.readline().strip():
        raise ValueError("File does not contain a valid OFF header")

    n_verts, n_faces, _ = tuple([int(s) for s in file.readline().strip().split(' ')])

    verts = [[float(s) for s in file.readline().strip().split(' ')] for _ in range(n_verts)]
    faces = [[int(s) for s in file.readline().strip().split(' ')][1:] for _ in range(n_faces)]

    return verts, faces

### Visualizing

It is generally useful to look at the data we are dealing with, so let's add a function for visualizing meshes, add another one for visualizing point clouds, and then visualize one example.

In [ ]:
def visualize_object(data):
    x_eye, y_eye, z_eye = 1.25, 1.25, 0.8
    frames = []

    def rotate_z(x, y, z, theta):
        w = x + 1j * y
        return np.real(np.exp(1j * theta) * w), np.imag(np.exp(1j * theta) * w), z

    for t in np.arange(0, 10.26, 0.1):
        xe, ye, ze = rotate_z(x_eye, y_eye, z_eye, -t)
        frames.append(
            {
                "layout": {
                    "scene": {
                        "camera": {
                            "eye": {"x": xe, "y": ye, "z": ze},
                        },
                    },
                },
            }
        )

    return go.Figure(data=data, frames=frames)


def visualize_mesh(x, y, z, i, j, k):
    data = [go.Mesh3d(x=x, y=y, z=z, color="lightpink", opacity=0.50, i=i, j=j, k=k)]
    fig = visualize_object(data)
    fig.show()


def visualize_points(xs, ys, zs):
    data = [go.Scatter3d(x=xs, y=ys, z=zs, mode="markers")]
    fig = visualize_object(data)
    fig.update_traces(
        marker=dict(size=2, line=dict(width=2, color="DarkSlateGrey")),
        selector=dict(mode="markers"),
    )
    fig.show()

With the functions for visualization out of the way, let's load one example and visualize that.

In [ ]:
with open(dataset_root / "bed/train/bed_0001.off", "r") as f:
    verts, faces = read_off(f)

i, j, k = np.array(faces).T
x, y, z = np.array(verts).T

In [ ]:
visualize_mesh(x, y, z, i, j, k)

This mesh definitely looks like a bed. Let us now visualize only the vertices of the mesh without the faces.

In [ ]:
visualize_points(x, y, z)

Unfortunately, the vertices do not look like a bed. Actually, flat parts of a surface don’t require any points for mesh construction. That’s why points are primarily located at angles and rounded parts of the bed. As points are not uniformly distributed across object’s surface, it could be difficult for our PointNet to classify them. Especially knowing that this point cloud doesn’t even look like a bed.

### Transforms

#### Point Sampling

As we want it to look more like a real bed, let's use a function to sample points on the surface uniformly. We will implement this as a torchvision transform for ease of use.

In [ ]:
class PointSampler(v2.Transform):
    def __init__(self, num_points: int):
        super().__init__()

        self.num_points = num_points

    def _triangle_area(self, pt1, pt2, pt3):
        side_a = np.linalg.norm(pt1 - pt2)
        side_b = np.linalg.norm(pt2 - pt3)
        side_c = np.linalg.norm(pt3 - pt1)

        s = 0.5 * (side_a + side_b + side_c)

        return max(s * (s - side_a) * (s - side_b) * (s - side_c), 0) ** 0.5

    def _sample_point(self, pt1, pt2, pt3):
        # barycentric coordinates on a triangle
        # https://mathworld.wolfram.com/BarycentricCoordinates.html

        s, t = sorted([random.random(), random.random()])
        f = lambda i: s * pt1[i] + (t - s) * pt2[i] + (1 - t) * pt3[i]

        return (f(0), f(1), f(2))

    def forward(self, inpt):
        verts, faces = inpt
        verts = np.array(verts)
        areas = np.zeros((len(faces),))

        for i in range(len(areas)):
            areas[i] = self._triangle_area(
                verts[faces[i][0]], verts[faces[i][1]], verts[faces[i][2]]
            )

        sampled_faces = random.choices(
            faces, weights=areas, cum_weights=None, k=self.num_points
        )

        sampled_points = np.zeros((self.num_points, 3))

        for i in range(len(sampled_faces)):
            sampled_points[i] = self._sample_point(
                verts[sampled_faces[i][0]],
                verts[sampled_faces[i][1]],
                verts[sampled_faces[i][2]],
            )

        return torch.from_numpy(sampled_points).to(dtype=torch.float32)

Now let's try this out and make sure it works.

In [ ]:
sampler = PointSampler(num_points=3000)
points = sampler((verts, faces))

visualize_points(*points.T)

Success! This pointcloud looks much more like a bed.

#### Normalization

We know that objects can have different sizes and can be placed in different parts of our coordinate system. Therefore we need to normalize the point clouds.

**TODO:** Implement the transform to normalize the point cloud such that it is centered at the origin and all points lie within a unit sphere, i.e. the maximum distance of a point to the origin is 1.

In [ ]:
class Normalize(v2.Transform):
    def forward(self, points: torch.Tensor) -> torch.Tensor:
        """
        Args:
          points: Point cloud as torch.Tensor of shape (number_of_points, 3).

        Returns:
          points: Normalized point cloud as torch.Tensor of shape (number_of_points, 3).
        """

        # TODO START ################
        # Implement point cloud normalization.
        # TODO END ##################

        return points

Let's now test and visualize this.

In [ ]:
normalize = Normalize()
points_norm = normalize(points)

visualize_points(*points_norm.T)

Notice how the axis limits have changed.

#### Augmentations: Random Rotation

We will implement two data augmentations that 1) randomly rotate our point cloud and 2) add noise to it to prevent overfitting.

**TODO:** Implement the transform to randomly rotate the whole point cloud around the z-axis (last axis). To do so, sample an random angle between 0° and 360°, create the rotation matrix and transform the point cloud with the rotation matrix. You can visualize the result below. Hint: https://en.wikipedia.org/wiki/Rotation_matrix#Basic_3D_rotations

In [ ]:
class RandomRotateZ(v2.Transform):
    def forward(self, points: torch.Tensor) -> torch.Tensor:
        """
        Args:
          points: Point cloud as torch.Tensor of shape (number_of_points, 3).

        Returns:
          points: Randomly rotated point cloud as torch.Tensor of shape (number_of_points, 3).
        """
        # TODO START ################
        # Implement random rotation around z axis.
        # TODO END ##################

        return points

Let's again test and visualize this.

In [ ]:
random_rotate = RandomRotateZ()
points_rot = random_rotate(points_norm)

visualize_points(*points_rot.T)

#### Augmentations: Random Noise

**TODO:** Implement the transform to add Gaussian noise with a standard deviation of 0.02 to each point separately.

In [ ]:
class RandomNoise(v2.Transform):
    def __init__(self, std: float = 0.02):
        super().__init__()

        self.std = std

    def forward(self, points: torch.Tensor) -> torch.Tensor:
        """
        Args:
          points: Point cloud as torch.Tensor of shape (number_of_points, 3).

        Returns:
          points: Noisy point cloud as torch.Tensor of shape (number_of_points, 3).
        """
        # TODO START ################
        # Add random Gaussian noise.
        # TODO END ##################

        return points

And let's again visualize the results.

In [ ]:
random_noise = RandomNoise(std=0.02)
points_noisy = random_noise(points_rot)

visualize_points(*points_noisy.T)

You should now see a noisy, randomly rotated bed.

And with that, we have all the data transforms and augmentations we need for now.

### Dataset and Dataloaders

Now we can create our training and validation datasets. For this, we first need a dataset class that provides our point cloud data.

In [ ]:
class PointCloudDataset(Dataset):
    def __init__(self, root_dir: Path | str, transform: v2.Transform | None = None, split: str = "train"):
        super().__init__()

        self.root_dir = Path(root_dir)
        self.transform = transform

        subdirs = [x for x in self.root_dir.iterdir() if x.is_dir()]
        classes = {folder: i for i, folder in enumerate(subdirs)}

        files = []
        for dir in subdirs:
            new_dir = dir / split
            category = dir.stem

            for file in new_dir.iterdir():
                if file.suffix == ".off":
                    sample = {}
                    sample["pcd_path"] = file
                    sample["category"] = category

                    files.append(sample)

        self.classes = {c.stem: i for c, i in classes.items()}
        self.files = files

    def __len__(self):
        return len(self.files)

    def _load_and_process(self, file):
        verts, faces = read_off(file)

        if self.transform is not None:
            points = self.transform((verts, faces))

        return points

    def __getitem__(self, idx):
        pcd_path = self.files[idx]["pcd_path"]
        category = self.files[idx]["category"]

        with open(pcd_path, "r") as f:
            points = self._load_and_process(f)

        return {"points": points, "category": self.classes[category]}

Next, we can define transforms and with those the datsets for training and validation. We sample 1024 points per cloud as in the paper in order to batch our samples.

In [ ]:
train_transforms = v2.Compose(
    [
        PointSampler(1024),
        Normalize(),
        RandomRotateZ(),
        RandomNoise(),
    ]
)

val_transforms = v2.Compose(
    [
        PointSampler(1024),
        Normalize(),
    ]
)

train_ds = PointCloudDataset(dataset_root, transform=train_transforms, split="train")
valid_ds = PointCloudDataset(dataset_root, transform=val_transforms, split="test")

Now let's show some basic statistics for the data:

In [ ]:
inv_class_map = {i: cat for cat, i in train_ds.classes.items()}

print("Classes:", inv_class_map)
print("Training dataset size:", len(train_ds))
print("Validation dataset size:", len(valid_ds))
print("Number of classes:", len(train_ds.classes))

# let's also load a random sample
sample_index = 42
sample = train_ds[sample_index]
points = sample["points"]
category = sample["category"]

print(f"Sample {sample_index} has {points.shape[0]} points and category {inv_class_map[category]}")

Finally, we can build our dataloaders.

In [ ]:
train_loader = DataLoader(dataset=train_ds, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(dataset=valid_ds, batch_size=batch_size)

## Model

Let's implement PointNet now.

We will first implement the T-Net, which predicts a transformation to transform the point cloud and later in the network the point features. It takes the point cloud as input and predicts a 3x3 transformation matrix that
is then multiplied with the point cloud to transform each point of the point cloud. Thus it learns automatically to align the data. Its architecture consists of a small PointNet and we initilize the transformation matrix to be the identity by default to start training with no transformations at all. So, we just add an identity matrix to the transformation.

**TODO:** Construct the layers of the T-Net and implement its forward pass. It has the following layers, where FC stands for fully-connected (`nn.Linear`) layer (named MLP in the paper):

1.   Shared FC(64), BatchNorm, ReLU.
2.   Shared FC(128), BatchNorm, ReLU.
3.   Shared FC(1024), BatchNorm, ReLU.
4.   Max Pooling to aggregate all point features to one global feature vector.
5.   FC(512), BatchNorm, ReLU.
6.   FC(256), BatchNorm, ReLU.
7.   FC(k*k).

The numbers in the brackets are the number of output channels of each FC layer. You can implement the shared FC by using a 1D 1x1 convolution layer (`nn.Conv1d`).

In [ ]:
class TNet(nn.Module):
    """
    T-Net

    Args:
       k: dimension of the transformation matrix kxk
    """

    def __init__(self, k: int = 3):
        super().__init__()

        self.k = k

        # TODO START ################
        # Initialize network submodules.
        # TODO END ##################

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
          x (batch_size, 3, number_of_points)

        Returns:
          matrix (batch_size, k, k)
        """
        bs, _3, n = x.shape

        # TODO START ################
        # Apply network submodules to transform x.
        # TODO END ##################

        # initialize as identity
        init = torch.eye(self.k, requires_grad=True).repeat(bs, 1, 1)
        if x.is_cuda:
            init = init.cuda()

        matrix = x.view(-1, self.k, self.k) + init

        return matrix

Next, we will implement PointNet until the the global feature extraction.

**TODO:** Construct the layers of PointNet until the global feature extraction and implement the forward pass to get the global feature vector. After obtaining the transformation matrix from a T-Net, multiply it with the point cloud to transform the point cloud. You can use torch.bmm() or torch.matmul() for that. Return the global feature vector and both T-Nets transformation matrices of in the forward function.

This network has the following layers:
1. TNet(k=3)
2. shared FC(64), BatchNorm, ReLU
3. TNet(k=64)
4. shared FC(128), BatchNorm, ReLU
5. shared FC(1024), BatchNorm
6. Max Pooling over the points dimension to aggregate them all to one global feature vector.

In [ ]:
class Transform(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO START ################
        # Initialize network submodules.
        # TODO END ##################

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
          x (batch_size, 3, number_of_points)

        Returns:
          output (batch_size, 1024), matrix3x3 (batch_size, 3, 3), matrix64x64 (batch_size, 64, 64)
        """
        # TODO START ################
        # Apply network submodules to transform x and compute matrix3x3 and matrix64x64.
        matrix3x3 = None
        matrix64x64 = None
        # TODO END ##################

        return x, matrix3x3, matrix64x64

Now we wrap the Tranform module into a PointNet module with the FC head on top and dropout in between.

In [ ]:
class PointNet(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()

        self.transform = Transform()

        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)

        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)

        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)

        x, matrix3x3, matrix64x64 = self.transform(x)

        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.dropout(self.fc2(x))))

        x = self.fc3(x)

        return x, matrix3x3, matrix64x64

We use the standard `CrossEntropyLoss()` for classification and add the regularization loss to make the transformation matrices close to orthogonal.

In [ ]:
class PointNetLoss(nn.Module):
    def __init__(self, alpha: float = 0.0001):
        super().__init__()

        self.alpha = alpha
        self.criterion = nn.CrossEntropyLoss()

    def forward(
        self,
        outputs: torch.Tensor,
        labels: torch.Tensor,
        m3x3: torch.Tensor,
        m64x64: torch.Tensor,
    ) -> torch.Tensor:
        bs, *_ = outputs.shape

        id3x3 = torch.eye(3, device=outputs.device).repeat(bs, 1, 1)
        id64x64 = torch.eye(64, device=outputs.device).repeat(bs, 1, 1)

        diff3x3 = id3x3 - torch.bmm(m3x3, m3x3.transpose(1, 2))
        diff64x64 = id64x64 - torch.bmm(m64x64, m64x64.transpose(1, 2))

        cls_loss = self.criterion(outputs, labels)
        reg_loss = (torch.norm(diff3x3) + torch.norm(diff64x64)) / bs

        return cls_loss + self.alpha * reg_loss

Since we use PyTorch Lightning, we now need to define a `LightningModule` that wraps everything and sets up our optimizer and other parts, like training and evaluation loops. In essence, the `LightningModule` is an extension to the base PyTorch `Module`.

In particular, we define:
- A constructor, which sets up our model architecture just like we would in any plain custom PyTorch `Module`. Here, we just add our `PointNet` and loss.
- The `configure_optimizers()` function, which configures and returns the optimizer to use.
- The `forward()` function, which again acts just like the plain `Module`'s forward function.
- The `training_step()` function, which performs one iteration of the training loop.
- The `validation_step()` function, which performs one iteration of the validation/evaluation loop.
- The `predict_step()` function, which performs one iteration of the prediction loop. Note that this is similar to the validation function, but it does not compute any metrics and just returns the predicted outputs (and in our case also ground-truth labels).

In [ ]:
class PointNetModel(L.LightningModule):
    def __init__(
        self,
        num_classes=10,
        lr=1e-3,
    ):
        """
        The constructor. Here we define our model architecture.

        Args:
            num_classes (int): The number of classes.
            lr (float): The learning rate.
        """

        super().__init__()

        # This saves any input arguments as hyperparameters (self.hparams) and
        # automatically makes them accessible to Weights and Biases. So you can
        # easily see them in the wandb dashboard.
        self.save_hyperparameters()

        self.num_classes = num_classes
        self.lr = lr

        self.module = PointNet(num_classes=num_classes)
        self.loss = PointNetLoss()
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

    def configure_optimizers(self):
        """
        Construct and configure the optimizer used for optimization.
        """
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

    def forward(self, x):
        """
        The forward() function. Just like standard PyTorch. Here we perform our
        model forward pass.
        """
        return self.module(x)

    def training_step(self, batch, batch_idx):
        """
        The steps needed for training. This function essentially contains the
        part of the training loop that runs the actual model and computes the
        loss and any metrics based on its prediction. The function returns the
        loss, which is then used by lightning to perform back-propagation
        (meaning you don't need to call backward() and run the optimizer
        yourself).
        """
        points, labels = batch["points"], batch["category"]

        # run the model
        outputs, m3x3, m64x64 = self.module(points)

        # compute the loss
        loss = self.loss(outputs, labels, m3x3, m64x64)

        # log the loss
        self.log("train_loss", loss)

        return loss

    def validation_step(self, batch, batch_idx):
        """
        The core steps to be run in the validation loop. This basically just
        runs our model and computes and logs its accuracy given a pair of input
        samples. Any metrics we log here are (by default) automatically
        accumulated and averaged over the entire validation set.
        """
        points, labels = batch["points"], batch["category"]

        # run the model
        outputs, m3x3, m64x64 = self.module(points)

        # compute the loss
        loss = self.loss(outputs, labels, m3x3, m64x64)

        # compute the accuracy
        acc = self.accuracy(preds=outputs, target=labels)

        # log loss and accuracy
        self.log("val_loss", loss)
        self.log("val_accuracy", acc)

    def predict_step(self, batch, batch_idx):
        """
        Base function to predict emebddings for all samples in the batch.
        """
        points, labels = batch["points"], batch["category"]

        # run the model
        outputs, _, _ = self.module(points)

        # compute the predictions
        _, preds = torch.max(outputs, dim=1)

        return preds, labels


## Training

We will now train our PointNet model for 1 epoch. If your implementation is correct, the loss should reduce quickly and you should get an validation accuracy of around 50% - 70%.

Normally we would train the model for 15 epochs. However, to save time, we just train for one epoch, which already gives decent results.

**Note:** You may want to set `accelerator="cpu"` while implementing/debugging your code. That way you can save GPU compute on colab and only switch the runtime to GPU once you have made sure your code is running.

In [ ]:
model = PointNetModel()

trainer = build_trainer(max_epochs=1, accelerator="gpu")
trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=valid_loader)
trainer.validate(model=model, dataloaders=valid_loader)

wandb.finish()

## Testing

Finally, let's evaluate our trained model by plotting a confusion matrix.

For this, we will first generate and collect all predictions and corresponding labels on the validation dataset.

In [ ]:
trainer = build_trainer(log=False)

# Note: PyTorch lightning automatically saves checkpoints after each epoch
# (under the ./logs directory). If you have already trained your model, you can
# set the checkpoint path to load it here.
preds = trainer.predict(model=model, dataloaders=valid_loader, ckpt_path=None)

preds, labels = zip(*preds)

labels = np.concat([x.numpy() for x in labels])
preds = np.concat([x.numpy() for x in preds])

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, preds)
cm

In [ ]:
import itertools
import matplotlib.pyplot as plt


def plot_confusion_matrix(
    cm, classes, normalize=False, title="Confusion matrix", cmap=plt.cm.Blues
):
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print("Confusion matrix, without normalization")

    plt.imshow(cm, interpolation="nearest", cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(
            j,
            i,
            format(cm[i, j], fmt),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
        )

    plt.tight_layout()
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

In [ ]:
plt.figure(figsize=(8,8))
plot_confusion_matrix(cm, [inv_class_map[i] for i in range(10)], normalize=True)

In [ ]:
plt.figure(figsize=(8,8))
plot_confusion_matrix(cm, [inv_class_map[i] for i in range(10)], normalize=True)

**TODO:** Which classes are frequently predicted incorrectly? Which classes look similar for the network?

Answer: